In [1]:
import pandas as pd
data=pd.read_csv("fanamc_articles.csv")

In [27]:
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html

df = pd.DataFrame({
    "Fruit": ["Apple", "Orange", "Banana"],
    "Sales": [25, 18, 35]
})


fig = px.bar(df, x="Fruit", y="Sales")

app = Dash(__name__)

app.layout = html.Div([
    html.H1("Fruit Sales"),
    dcc.Graph(figure=fig)
])

if __name__ == "__main__":
    app.run(debug=True)


OSError: Address 'http://127.0.0.1:8050' already in use.
    Try passing a different port to run.

In [26]:
from dash import Dash, dcc, html, Input, Output, callback, no_update
from dash.exceptions import PreventUpdate

external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css']

app = Dash(__name__, external_stylesheets=external_stylesheets)

app.layout = html.Div([
    html.P('Enter a composite number to see its prime factors'),
    dcc.Input(id='num', type='number', debounce=True, min=2, step=1),
    html.P(id='err', style={'color': 'red'}),
    html.P(id='out')
])

@callback(
    Output('out', 'children'),
    Output('err', 'children'),
    Input('num', 'value')
)
def show_factors(num):
    if num is None:
        # PreventUpdate prevents ALL outputs updating
        raise PreventUpdate

    factors = prime_factors(num)
    if len(factors) == 1:
        # dash.no_update prevents any single output updating
        # (note: it's OK to use for a single-output callback too)
        return no_update, '{} is prime!'.format(num)

    return '{} is {}'.format(num, ' * '.join(str(n) for n in factors)), ''

def prime_factors(num):
    n, i, out = num, 2, []
    while i * i <= n:
        if n % i == 0:
            n = int(n / i)
            out.append(i)
        else:
            i += 1 if i == 2 else 2
    out.append(n)
    return out

if __name__ == '__main__':
    app.run(debug=True)

In [9]:
import json
from dash import Dash, html, Input, Output, callback, ctx

app = Dash()

app.layout = html.Div([
    html.Button('Button 1', id='btn-1'),
    html.Button('Button 2', id='btn-2'),
    html.Button('Button 3', id='btn-3'),
    html.Div(id='container')
])


@callback(Output('container', 'children'),
              Input('btn-1', 'n_clicks'),
              Input('btn-2', 'n_clicks'),
              Input('btn-3', 'n_clicks'))
def display(btn1, btn2, btn3):
    if not ctx.triggered_id:
        button_id = 'No clicks yet'
    else:
        button_id = ctx.triggered_id

    ctx_msg = json.dumps({
        'states': ctx.states,
        'triggered': ctx.triggered,
        'inputs': ctx.inputs
    }, indent=2)

    return html.Div([
        html.Table([
            html.Tr([html.Th('Button 1'),
                     html.Th('Button 2'),
                     html.Th('Button 3'),
                     html.Th('Most Recent Click')]),
            html.Tr([html.Td(btn1 or 0),
                     html.Td(btn2 or 0),
                     html.Td(btn3 or 0),
                     html.Td(button_id)])
        ]),
        html.Pre(ctx_msg)
    ])


if __name__ == '__main__':
    app.run(debug=True)

In [10]:
from dash import Dash, html, Output, Input, State, no_update

app = Dash(suppress_callback_exceptions=True)

app.layout = html.Div([
    html.Div([
        html.Button(id="optional-inputs-button-1", children="Button 1", className="button"),
        html.Div(id="optional-inputs-container"),
        html.Div(id="optional-inputs-output", className="output")
    ], className="container")
])

@app.callback(
    Output("optional-inputs-container", "children"),
    Input("optional-inputs-button-1", "n_clicks"),
    State("optional-inputs-container", "children"),
    prevent_initial_call=True
)
def add_second_button(_, current_children):
    if not current_children:
        return html.Button(id="optional-inputs-button-2", children="Button 2", className="button")
    return no_update

@app.callback(
    Output("optional-inputs-output", "children"),
    Input("optional-inputs-button-1", "n_clicks"),
    Input("optional-inputs-button-2", "n_clicks", allow_optional=True),
    prevent_initial_call=True
)
def display_clicks(n_clicks1, n_clicks2):
    return f"Button 1 clicks: {n_clicks1} - Button 2 clicks: {n_clicks2}"

if __name__ == '__main__':
    app.run(debug=True)

In [11]:
from dash import Dash, dcc, html, Input, Output 
 
import plotly.express as px 
app = Dash(__name__) 
app.layout = html.Div([ 
    html.H1('Restaurant tips by day of week'), 
    dcc.Dropdown( 
        id="dropdown", 
        options=['Mon', "Fri", "Sat", "Sun"], 
        value="Fri", 
        clearable=False, 
    ), 
    dcc.Graph(id="graph"), 
]) 
@app.callback( 
    Output("graph", "figure"), 
    Input("dropdown", "value")) 
def update_bar_chart(day): 
    df = px.data.tips() # replace with your own data source 
    mask = df["day"] == day 
    fig = px.bar(df[mask], x="sex", y="total_bill", color="smoker", barmode="group") 
    return fig 
app.run(debug=True)

In [12]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import time

BASE_URL = "https://www.fanamc.com/"

headers = {
    "User-Agent": "Mozilla/5.0"
}

# Get homepage
response = requests.get(BASE_URL, headers=headers)
response.encoding = "utf-8"

soup = BeautifulSoup(response.text, "html.parser")

# Collect article links
article_links = set()

for a in soup.find_all("a", href=True):
    href = urljoin(BASE_URL, a["href"])

    if "/202" in href:      # Most Fana articles contain the publication year
        article_links.add(href)

print(f"Found {len(article_links)} articles")

data = []

for url in article_links:

    try:
        r = requests.get(url, headers=headers, timeout=10)
        r.encoding = "utf-8"

        article = BeautifulSoup(r.text, "html.parser")

        # ---------- Title ----------
        title = ""
        if article.find("h1"):
            title = article.find("h1").get_text(strip=True)

        # ---------- Date ----------
        date = ""

        time_tag = article.find("time")
        if time_tag:
            date = time_tag.get_text(" ", strip=True)

        # ---------- Category ----------
        category = ""

        cat = article.find("a", rel="category tag")
        if cat:
            category = cat.get_text(strip=True)

        # ---------- Language ----------
        html_tag = article.find("html")

        language = html_tag.get("lang", "") if html_tag else ""

        # ---------- Article ----------
        paragraphs = article.find_all("p")

        content = "\n".join(
            p.get_text(" ", strip=True)
            for p in paragraphs
        )

        if len(content) > 100:

            data.append({
                "Language": language,
                "Date": date,
                "Category": category,
                "Title": title,
                "Article": content,
                "URL": url
            })

        print(title)

        time.sleep(1)

    except Exception as e:
        print(e)

df = pd.DataFrame(data)

print(df.head())

df.to_csv(
    "fanamc_news.csv",
    index=False,
    encoding="utf-8-sig"
)

Found 12 articles
Page Not Found!
Page Not Found!
Page Not Found!
Page Not Found!
Page Not Found!
Page Not Found!
Page Not Found!
Page Not Found!
Page Not Found!
Page Not Found!
Page Not Found!
Page Not Found!
  Language Date Category            Title  \
0    en-US                Page Not Found!   
1    en-US                Page Not Found!   
2    en-US                Page Not Found!   
3    en-US                Page Not Found!   
4    en-US                Page Not Found!   

                                             Article  \
0  \nFanaMC - In a Speed of life!\n404\nWe're sor...   
1  \nFanaMC - In a Speed of life!\n404\nWe're sor...   
2  \nFanaMC - In a Speed of life!\n404\nWe're sor...   
3  \nFanaMC - In a Speed of life!\n404\nWe're sor...   
4  \nFanaMC - In a Speed of life!\n404\nWe're sor...   

                              URL  
0  https://www.fanamc.com/2021/04  
1  https://www.fanamc.com/2020/11  
2  https://www.fanamc.com/2020/09  
3  https://www.fanamc.com/2020/05  
4 

In [13]:
import pandas as pd
from dash import Dash, html, dcc, dash_table, Input, Output
import plotly.express as px

# Read scraped data
df = pd.read_csv("fanamc_news.csv")

# Replace missing values
df.fillna("", inplace=True)

app = Dash(__name__)

app.layout = html.Div([

    html.H1(
        "Fana Media News Dashboard",
        style={"textAlign": "center"}
    ),

    html.Br(),

    dcc.Dropdown(
        id="category",
        options=[
            {"label": c, "value": c}
            for c in sorted(df["Category"].unique())
            if c != ""
        ],
        placeholder="Select Category"
    ),

    html.Br(),

    dcc.Input(
        id="search",
        placeholder="Search title...",
        style={"width": "100%"}
    ),

    html.Br(),
    html.Br(),

    dcc.Graph(id="category_graph"),

    html.Br(),

    dash_table.DataTable(
        id="table",
        columns=[
            {"name": i, "id": i}
            for i in df.columns
        ],
        page_size=10,
        style_table={
            "overflowX": "auto"
        },
        style_cell={
            "textAlign": "left",
            "whiteSpace": "normal",
            "height": "auto"
        }
    )

])


@app.callback(

    Output("table", "data"),
    Output("category_graph", "figure"),

    Input("category", "value"),
    Input("search", "value")

)
def update(category, search):

    filtered = df.copy()

    if category:
        filtered = filtered[
            filtered["Category"] == category
        ]

    if search:
        filtered = filtered[
            filtered["Title"].str.contains(
                search,
                case=False,
                na=False
            )
        ]

    fig = px.bar(
        filtered["Category"].value_counts().reset_index(),
        x="Category",
        y="count",
        title="Articles by Category"
    )

    return filtered.to_dict("records"), fig


if __name__ == "__main__":
    app.run(debug=True)

C:\Users\SOLOMON\AppData\Local\Temp\ipykernel_13636\1265296063.py:9: FutureWarning:

Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.



In [14]:
import requests
from bs4 import BeautifulSoup
import csv
import json
from datetime import datetime

# Target base URL
BASE_URL = "https://www.fanamc.com/"

# Set standard user-agent header to avoid getting blocked by the server
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

def scrape_fana_homepage(language_route=""):
    """
    Scrapes featured articles from the Fana MC page.
    language_route: Leave empty for default (Amharic), or use 'en/', 'ao/', 'tg/' depending on sub-domain structures.
    """
    target_url = f"{BASE_URL}{language_route}"
    print(f"Fetching articles from: {target_url}...")
    
    try:
        response = requests.get(target_url, headers=HEADERS, timeout=15)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Error connecting to the site: {e}")
        return []

    soup = BeautifulSoup(response.content, 'html.parser')
    scraped_data = []

    # Detect page language from the <html> tag attributes or route
    page_lang = soup.find('html').get('lang', 'am') if soup.find('html') else 'am'
    if language_route:
        page_lang = language_route.strip('/')

    # Fana uses standard article semantic blocks for posts
    articles = soup.find_all('article')
    
    # Fallback to common post container classes if semantic tags aren't cleanly isolated
    if not articles:
        articles = soup.find_all('div', class_=['post-item', 'td-block-span12', 'mag-box-item'])

    for article in articles:
        try:
            # 1. Extract Title
            title_tag = article.find(['h2', 'h3', 'h1'])
            if not title_tag:
                continue
            title = title_tag.get_text(strip=True)
            
            # 2. Extract Link to get full article text later
            link_tag = title_tag.find('a') or article.find('a')
            article_url = link_tag['href'] if link_tag and link_tag.has_attr('href') else None

            # 3. Extract Category
            #category_tag = article.find('a', class_=['category', 'post-category', 'td-post-category'])
            #category = category_tag.get_text(strip=True) if category_tag else "Uncategorized"

            # 4. Extract Date
            date_tag = article.find(['span', 'time', 'div'], class_=['date', 'post-date', 'entry-date'])
            date_str = date_tag.get_text(strip=True) if date_tag else datetime.today().strftime('%b %d, %Y')

            # 5. Extract Summary/Snippet (acts as placeholder if you don't deep-dive the link)
            snippet_tag = article.find(['p', 'div'], class_=['excerpt', 'post-excerpt', 'entry-content'])
            snippet = snippet_tag.get_text(strip=True) if snippet_tag else ""

            # Standardized payload
            item_data = {
                "language": page_lang,
                "date": date_str,
                "category": category,
                "title": title,
                "article_summary": snippet,
                "source_url": article_url
            }
            
            # Avoid duplicating layout blocks
            if item_data not in scraped_data:
                scraped_data.append(item_data)
                
        except Exception as e:
            # Skip corrupted elements cleanly
            continue

    return scraped_data

def save_to_csv(data, filename="fana_scraped_data.csv"):
    if not data:
        print("No data found to save.")
        return
        
    keys = data[0].keys()
    with open(filename, mode='w', encoding='utf-8', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=keys)
        writer.writeheader()
        writer.writerows(data)
    print(f"Successfully saved {len(data)} rows to {filename}")

# --- Execution ---
if __name__ == "__main__":
    # Scrape the main homepage
    results = scrape_fana_homepage()
    
    # Output to console cleanly
    print(json.dumps(results[:3], indent=2, ensure_ascii=False)) 
    
    # Export the dataset
    save_to_csv(results)

Fetching articles from: https://www.fanamc.com/...
[
  {
    "language": "en-US",
    "date": "Jun 26, 2026",
    "category": "Uncategorized",
    "title": "የሚኒስትሮች ምክር ቤት በዛሬው ዕለት የተለያዩ ውሳኔዎችን አሳለፈ",
    "article_summary": "",
    "source_url": "https://www.fanamc.com/archives/317557"
  },
  {
    "language": "en-US",
    "date": "Jun 26, 2026",
    "category": "Uncategorized",
    "title": "የሻዕቢያ ዋና አላማ የኢትዮጵያ አኩራፊ ሃይሎችን በማደራጀት በሀገሪቱ ሰላም እንዳይኖር ማድረግ ነው – አቶ ጌታቸው ረዳ",
    "article_summary": "",
    "source_url": "https://www.fanamc.com/archives/317548"
  },
  {
    "language": "en-US",
    "date": "Jun 26, 2026",
    "category": "Uncategorized",
    "title": "ኢትዮጵያ በለውጡ ማግስት ስኬታማ የቴክኖሎጂና የኢኮኖሚ ማሻሻያ አድርጋለች –…",
    "article_summary": "",
    "source_url": "https://www.fanamc.com/archives/317545"
  }
]
Successfully saved 13 rows to fana_scraped_data.csv


In [25]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd

BASE_URL = "https://www.fanamc.com/"
TECH_URL = "https://www.fanamc.com/%e1%89%a2%e1%8b%9d%e1%8a%90%e1%88%b5"

#TECH_URL =str(input("Enter url to scrap the web:"))
headers = {
    "User-Agent": "Mozilla/5.0"
}

# Get technology page
response = requests.get(TECH_URL, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

# Collect article links
links = set()

for a in soup.find_all("a", href=True):
    href = a["href"]

    if "https://www.fanamc.com/archives/" in href:
        links.add(urljoin(BASE_URL, href))

print(f"Found {len(links)} articles")
#href="https://www.fanamc.com/english/council-of-ministers-endorses-new-development-bank-accession-bill-among-key-financing-and-policy-decisions/"
data = []

# Visit each article
for link in links:
    try:
        article_page = requests.get(link, headers=headers)
        article_soup = BeautifulSoup(article_page.text, "html.parser")

        title = article_soup.find(class_="post-title" )
        title = title.get_text(strip=True) if title else "No title"
        time = article_soup.find(class_="post-published updated")
        time = time.get_text(strip=True)
        author=article_soup.find(class_="post-author author")
        author = author.get_text(strip=True) if author else "No author"
        paragraphs = article_soup.find_all(class_="entry-content clearfix single-post-content")
        
        content = " ".join(
            p.get_text(strip=True)
            for p in paragraphs
        )

        data.append({
               "language": "Amaric",
                "date": time,
                "category": "ቢዝነስ",
                "title": title,
                "article_summary": content,
                "Author":author,
                "source_url": link
        })
        
        print(title,":\n","Time:",time,"\n","link:",link,":\n", content,":\n",author)


    except Exception as e:
        print("Error:", e)

df = pd.DataFrame(data)
df.to_csv("fana_amharic_ቢዝነስ.csv", index=False)
#df.to_csv("bbc_technolo_newsgy.txt", index=False)

print("Saved!")

Found 44 articles
በአማራ ክልል ከ180 ሺህ በላይ ለሚሆኑ ዜጎች የሥራ ዕድል የፈጠረው የኮሪደር ልማት… :
 Time: OnJun 17, 2026 
 link: https://www.fanamc.com/archives/316930#respond :
 አዲስ አበባ፣ ሰኔ 10፣ 2018 (ኤፍ ኤም ሲ) በአማራ ክልል እየተከናወነ በሚገኛው የኮሪደር ልማት ፕሮጀክት እስካሁን ባለው ጊዜ ውስጥ ከ180 ሺህ በላይ ለሚሆኑ ዜጎች የሥራ ዕድል ተፈጥሯል።በ2018 በጀት ዓመት ክልሉን ዘመናዊና ለኑሮ ተስማሚ ለማድረግ የታቀዱ የኮሪደር ልማት ሥራዎች በ7 ሪጂኦፖሊታን የከተማ አስተዳደሮች እና ከ35 በላይ በሚሆኑ መካከለኛና አነስተኛ ከተሞች ላይ በከፍተኛ ፍጥነትና ስኬት እየተከናወኑ ይገኛሉ።በመሠረተ ልማት ዘርፍ በበጀት ዓመቱ ከ204 ኪሎ ሜትር በላይ መንገድ ለመሥራት ታቅዶ እስካሁን 137 ኪሎ ሜትር በላይ የሚረዝም መንገድ ማጠናቀቅ መቻሉን የክልሉ ከተማና መሠረተ ልማት ቢሮ መረጃ አመልክቷል፡፡በዚህም ከተከናወኑት የሥራ ክፍሎች መካከል 18 ኪሎ ሜትር የአስፋልት ንጣፍ መንገድ፣ 64 ኪሎ ሜትር የእግረኛ መንገድ እንዲሁም ከ54 ኪሎ ሜትር በላይ የሳይክል መስመር ግንባታ ይገኙበታል።የከተሞችን ገጽታ ለመቀየርና ማራኪ ለማድረግ በውበት ሥራ ከተያዘው 129 ሄክታር በላይ መሬት ውስጥ በአሁኑ ወቅት ከ104 ሄክታር በላይ የሚሆነውን ማልማት የተቻለ ሲሆን÷ ከዚህ ውስጥ የመንገድ ዳር አረንጓዴ ልማትንና የመናፈሻ ቦታዎችን ጨምሮ ከ54 ሄክታር በላይ መሬት ለምቷል።እንዲሁም 12 የሕዝብ መጸዳጃ ቤቶች፣ 20 የውኃ ፋውንቴን፣ 5 የአደባባይ ማስዋብ፣ 5 የልጆች መጫወቻ ቦታዎች፣ 8 የስፖርት ማዘውተሪያዎች እና 299 መለስተኛ ካፌዎች ተገንብተው ለአገልግሎት ዝግጁ ሆነዋል።ከተሞቹን ይበልጥ ዘመናዊና ደ

In [10]:
href

'https://www.fanamc.com/archives/category/fana_90'